# Bonus: Probability Basics

In this notebook, we have some content that was related to the first assignment in previous iterations of this class.
Part 1 looks at applications of Bayes' theorem, whereas part 2 walks through another modeling example related to coin flips.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

## Part 1: Testing Illnesses

Consider a dangerous and/or common illness that people are being tested for to recognize it early (e.g. cancer) 
and/or prevent its spread (e.g. COVID). 
The test is either positive or negative.
We make the following assumptions:

- About 1% of the population has the illness. 
That is, any given person has a 1% "a priori" probability of being sick.
- If a sick person is tested, the test returns a positive result 99.9% of the time.
- If a healthy person is tested, the test still returns a false positive result 1% of the time.

You take part in a study where a random, representative sample of the population is tested for the illness. 
Your test result is positive. 
What is the probability that you have the illness?

This requires a basic grasp of marginal and conditional probabilities as well as Bayes' theorem. 
These are fundamental concepts without which you will be lost in this class! 
[The corresponding wiki article](https://en.wikipedia.org/wiki/Bayes%27_theorem) should be sufficient.

For the simulation, we can generate uniform random numbers in the range [0, 1] via functions such as
`np.random.rand`, and such random numbers will be smaller than another fixed number `p` with probability `p`.
For example, the chance of the uniform random number being smaller than `p=0.9` is 90%.

We simulate multiple experiments.
That is, in each experiment, we collect `sample_size` many samples (i.e. people) and get the sick/test result proportions.
And this process in turn is repeated `n_experiments` many times, giving us different proportions each time.

This obviously wouldn't be doable in real life -- this is just to show how much variance there is between repetitions.
Results are unreliable!

Also see what happens when you decrease `sample_size`.

In [ ]:
sample_size = 10000
n_experiments = 1000

sick_prob = 0.01
test_prob_if_sick = 0.999
test_prob_if_healthy = 0.01

In [ ]:
all_probs = []

for experiment_ind in range(n_experiments):
    population_sickness = np.random.rand(sample_size)
    is_sick = population_sickness < sick_prob

    population_tests = np.random.rand(sample_size)
    sick_people = population_tests[is_sick]
    healthy_people = population_tests[np.logical_not(is_sick)]
    
    positive_and_sick = sick_people < test_prob_if_sick
    positive_and_healthy = healthy_people < test_prob_if_healthy
    
    denominator = sum(positive_and_healthy) + sum(positive_and_sick)
    if denominator > 0:
        prob_sick = sum(positive_and_sick) / denominator
    else:
        prob_sick = 0.
        
    all_probs.append(prob_sick)
    
all_probs = np.array(all_probs)

plt.hist(all_probs, bins=50)
plt.xlabel("False positive rate estimate")
plt.ylabel("Experiments")
plt.show()

Below is a faster, parallelized version, but it may be harder to understand than the loop above.
Basically we just create a `n_experiments x sample_size` array of probabilities and run everything at once.

In [ ]:
population_sickness = np.random.rand(n_experiments, sample_size)
is_sick = population_sickness < sick_prob
population_tests = np.random.rand(n_experiments, sample_size)

positive_and_sick = is_sick * (population_tests < test_prob_if_sick)
positive_and_healthy = np.logical_not(is_sick) * (population_tests < test_prob_if_healthy)

denominator = np.sum(positive_and_sick, axis=1) + np.sum(positive_and_healthy, axis=1)

prob_sick = np.sum(positive_and_sick, axis=1) / denominator

plt.hist(prob_sick, bins=50)
plt.xlabel("False positive rate estimate")
plt.ylabel("Experiments")
plt.show()

Even with 10000 people, we get quite a range of results (about 3% standard deviation):

In [ ]:
mean_prob = np.mean(all_probs)
stddev = np.std(all_probs)

print(mean_prob, stddev)

We can also solve this analytically.
For Bayes' rule, we end up having to compute $\frac{(0.999*0.01)}{(0.999*0.01 + 0.01*0.99)}$.
The result should be very close to the simluation, assuming you used plenty of samples.

In [ ]:
exact_result = 0.999*0.01 / (0.999*0.01 + 0.01*0.99)
print(exact_result)

The chance to be sick given a positive test is only 50%!

Next, assume the test result is negative. 
What is the probability that you have the illness anyway?

In [ ]:
converse = 0.001*0.01 / (0.001*0.01 + 0.99*0.99)
print(converse)
# 1e-5 ~ 0.00001 ~ 0.001%

This is very small -- only around 0.001%, or 1 in 100,000.

To bullet-proof the results of their study, the researchers decide to administer _two_ tests to each participant.
The second test has the following properties:
   - If a sick person is tested, the test returns a positive result 96% of the time.
   - If a healthy person is tested, the test still returns a positive result 2% of the time.  
     
As we can see, the second test is much more prone to errors than the first.
However, assume that the results of the second test are _conditionally independent_ of the first. 
That is, whether the second test makes an error does not depend on whether there is an error on the first test and 
vice versa, _given whether a person is sick or not_.

Now, _both_ of your tests come back positive. 
Given this information, what is the probability that you are indeed sick?

In [ ]:
two_tests = 0.999*0.96*0.01 / (0.999*0.96*0.01 + 0.01*0.02*0.99)

print(two_tests)

If we have to _independent_ tests, and they are both positive, our chance of being sick is very high (around 98%)!

## Part 2: A More Complex Coin Flip

In the exercise, we looked at how one can model a coin toss using a Bernoulli distribution, and using this model and
given some data, decide whether a coin is fair or not.

Here we extend this to a slightly more complex scenario:
Say I have a coin, and I flip it.
If it lands tails up, I flip again.
If it lands heads up, I stop.
Once it lands heads up, I record _the number of times I saw tails before I got heads_.
This is one trial.
Then I repeat this many more times to get more trials.
Given the results, do you think this is a fair coin?

We can proceed as follows:
- The appropriate probability distribution here is a 
[geometric distribution](https://en.wikipedia.org/wiki/Geometric_distribution).
Given a parameter `p`, this gives the probability of seeing a "success" (heads) after *exactly* `k` failures (tails).
   - **Be careful**, there is also another "version" of this distribution which gives the probability of success after exactly
  `k` *trials* (i.e. `k-1` failures and one success). 
  This is slightly different!
  Our data gives the number of failures, _without_ the success!
  If you mix this up, you will get slightly different results from the intended one (not a big deal).
- Derive the log probability for one trial and write the sum over trials -- this is the quantity we want to maximize.
- Use basic calculus to get the derivative of this quantity with respect to `p`.
- Now you have two options:
  1. Set the derivative to 0 and solve for `p`.
  2. Implement gradient descent to find the optimal `p` iteratively.

Either way, this should get you the `p` that best explains the data according to the maximum likelihood principle.
Is this close to 0.5?

You might want to review the derivation for the Bernoulli case in 
[our blog post on this topic](https://ovgu-ailab.github.io/blog/methods/2025/09/08/probabilistic-models.html).


Here we first generate the samples (in the actual assignment, you would have gotten this as a download) and have a look.
Obviously, we know `p` here, since we have to generate the data. But pretend for a moment you just get `samples` without knowing `p`. :)

In [ ]:
p = 0.7
n_trials = 10000
samples = np.random.geometric(p=p, size=n_trials) - 1 

In [ ]:
(samples == 0).mean()

In [ ]:
max_tosses = samples.max()
bins = np.arange(-0.5, max_tosses + 1.5)
plt.hist(samples, bins=bins, rwidth=0.8)
plt.xlabel("Number of tails until heads")
plt.ylabel("Occurrence")
plt.show()

Logarithmic y scale shows the data better:

In [ ]:
max_tosses = samples.max()
bins = np.arange(-0.5, max_tosses + 1.5)
plt.hist(samples, bins=bins, rwidth=0.8)
plt.yscale("log")
plt.xlabel("Number of tails until heads")
plt.ylabel("Occurrence")
plt.show()

You can mathematically derive the optimal parameter $\theta$ according to maximum likelihood -- this will be our "gold standard".
The solution is  $\frac{N}{N + \sum_i k_i}$, where `N` is the total number of trials (one trial is "one sequence of flips until heads"), and the sum is the total number of tails seen over all trials.

In [ ]:
best_p_guess = n_trials / (samples.sum() + n_trials)
best_p_guess

We can also compute the log-likelihood as well as the derivative. Note, the case $\theta = 1$ or $\theta = 0$ leads to ill-defined log-likelihoods, so we cut off the edges and only check from $\epsilon$ to $1 - \epsilon$ for a small number $\epsilon$.

We can see that the log-likelihood indeed maximizes at the value we see above, and the derivative is 0 there.

In [ ]:
# log likelihood as function of theta
eps = 0.05
theta_space = np.linspace(eps, 1-eps, 1000)

loglike = samples.sum() * np.log(1-theta_space) + len(samples) * np.log(theta_space)
plt.plot(theta_space, loglike, label="value")

plt.vlines(p, ymin=loglike.min(), ymax=loglike.max(), colors="black", linestyles="--", label=r"optimal $\theta$")

plt.title("log likelihood")
plt.xlabel(r"$\theta$")
plt.ylabel("Value")
plt.legend()
plt.show()

gradient = len(samples) / theta_space - samples.sum() / (1 - theta_space)
plt.plot(theta_space, gradient, label="gradient")
plt.hlines(np.zeros_like(theta_space), xmin=0, xmax=1, color="black", linestyle="--", label="zero")
plt.title("gradients")
plt.xlabel(r"$\theta$")
plt.ylabel("Value")
plt.legend()
plt.show()

We could even solve this through gradient descent. Unfortunately this is very unstable and highly dependent on the learing rate.

In [ ]:
guess_parameter = 0.5  # some initial value
n_steps = 100
learning_rate = 0.00002  # larger LR tends to break

for step in range(1, n_steps+1):
    gradient = len(samples) / guess_parameter - samples.sum() / (1 - guess_parameter)
    
    # decaying learning rate makes it more stable for higher learning rates
    # potential fix: clip this parameter to some epsilon > 0 to prevent accidentally jumping into negative values
    guess_parameter += learning_rate/step * gradient
    
# we print the resulting p
print(guess_parameter)

We can even do it with torch -- no need to compute gradients ourselves!

In [ ]:
import torch

In [ ]:
def ll(parameter):  # log likelihood
    return samples.sum() * torch.log(1-guess_parameter) + len(samples) * torch.log(guess_parameter)

guess_parameter = torch.tensor([0.5], requires_grad=True)
n_steps = 100
learning_rate = 0.00002

for step in range(1, n_steps + 1):
    loss = ll(guess_parameter)
    loss.backward()

    with torch.no_grad():
        guess_parameter += learning_rate/step * guess_parameter.grad
    guess_parameter.grad.zero_()
    
print(guess_parameter)